# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Our primary research question is: **How can we programmatically identify and rank decaying search content to prioritize editorial refreshes?**

This model supports the SEO and Content teams by replacing manual, heuristic-based content audits with a scalable machine learning pipeline. It predicts the probability of a page experiencing traffic decline, allowing the business to allocate editorial resources to high-risk, high-impact pages before they lose search visibility.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

This project utilizes the `content_refresh_anonymized.csv` dataset provided by the FlyRank internship.

* **Scope:** The data represents a snapshot of content performance metrics (impressions, clicks, sessions, CTR, position) over a 90-day window.
* **Exclusions:** To ensure the data is public-safe and strictly privacy-compliant, all Personally Identifiable Information (PII), client names, domain names, exact URLs, and raw search queries have been stripped.
* **Features used:** Anonymized structural and performance features (e.g., `word_count`, `impressions_90d`, `content_age_days`, `competition_level`).

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

Our approach upgrades standard linear classification to a production-grade Gradient Boosted Tree architecture.

* **Label Definition:** The target variable is `is_declining_label` (binary classification).
* **Baseline:** The pipeline measures against a fixed baseline heuristic (`baseline_refresh_score`) to mathematically prove ML lift.
* **Advanced Modeling:** We engineered an `XGBClassifier` optimized for imbalanced ranking (`scale_pos_weight=5`) and evaluated via `aucpr` (Area Under the Precision-Recall Curve).
* **Validation Design:** We implemented a strict **Client Holdout Split** (80/20). By stratifying on `client_id`, we ensured that no individual client's data leaked from the training set into the test set, guaranteeing real-world generalization.
* **Architectural Fixes:** The pipeline was heavily optimized by replacing slow row-wise Pandas operations with vectorized boolean masking, reducing evaluation latency to milliseconds.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Our custom `xgboost_advanced` model was evaluated against the baseline heuristic on the strict client holdout split. The measured ROC AUC and Precision@50 demonstrate a clear directional lift in identifying true declining content compared to the baseline rules.

In [ ]:
import pandas as pd
import json
from IPython.display import display

# Load the model results JSON we generated
with open("../../outputs/model_results.json", "r") as f:
    results = json.load(f)

# Build the honest comparison table
metrics = []
for model_name, model_metrics in results["models"].items():
    metrics.append({
        "Model": model_name,
        "Precision@50": model_metrics["precision_at_50"],
        "ROC AUC": model_metrics["roc_auc"],
        "Avg Precision": model_metrics["average_precision"]
    })

# Add Baseline
baseline = results["baseline"]
metrics.append({
        "Model": "baseline_rules",
        "Precision@50": baseline["baseline_precision_at_50"],
        "ROC AUC": baseline["baseline_roc_auc"],
        "Avg Precision": baseline["baseline_average_precision"]
})

df_results = pd.DataFrame(metrics).sort_values(by="Precision@50", ascending=False)
display(df_results)

## 5. Limitations

*What this work cannot claim.*

While this model provides strong directional signals, we must acknowledge the following limitations:
* **Decision-Support Only:** This pipeline is a decision-support tool, not an automated publishing engine. Human editorial review remains strictly required.
* **Missing Semantic Context:** Because the data is strictly anonymized for public safety (stripping URLs, keywords, and actual text), the model relies purely on structural and performance telemetry. It cannot measure the actual semantic quality of the written content.
* **Correlation vs Causation:** The feature importance scores indicate observed correlations with traffic decline, not guaranteed causal links.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
# Load the final ranked queue
df_queue = pd.read_csv("../../outputs/refresh_queue.csv")

# Display the top 10 actionable recommendations
display_cols = ["final_rank", "content_id", "confidence", "suggested_action", "final_refresh_score", "impressions_90d"]
top_recommendations = df_queue[display_cols].head(10)
display(top_recommendations)

Below is the top of our ranked action playbook. These represent the highest-priority, high-confidence rows identified by the model for editorial review.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
from IPython.display import SVG, display
import os

chart_dir = "../../outputs/charts/"
charts = [
    "top_feature_importance.svg",
    "action_mix.svg",
    "confidence_mix.svg"
]

for chart in charts:
    chart_path = os.path.join(chart_dir, chart)
    if os.path.exists(chart_path):
        print(f"--- {chart} ---")
        display(SVG(filename=chart_path))

The following visual artifacts were generated during the evaluation phase and represent the core visualizations for the deployed paper.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.